# Chapter 6: Composition of Sets

Companion notebook for **Chapter 6** of *Bernoulli Types: An Algebra of Approximate Computation*. Source files: `book/chapters/ch06/`.

Chapter 6 composes approximate sets. Once chapter 5 fixes the per-set rates $(\fprate, \fnrate)$, the next question is what happens to those rates under union, intersection, complement, and difference. The rate formulas come directly from the Boolean composition theorem of chapter 3, lifted to the per-set level.

This notebook:

1. Restates the Bloom filter class of chapter 5 (per-index salt hashing, scales to arbitrary $k$).
2. Builds two filters $\ASet{B}_1$ and $\ASet{B}_2$ over overlapping item sets at $m = 5000$, $k = 7$, $n_1 = n_2 = 500$.
3. Measures the **predicate-OR** union FPR and verifies it against the closed-form $1 - (1 - \fprate_1)(1 - \fprate_2)$ from \S6.1.
4. Measures the **bit-OR** union FPR and shows it sits *above* the predicate rate: the operational shortcut and the predicate composition are different events on the same query stream when the two filters share a hash family.
5. Measures the **bit-AND** intersection-shaped FPR and contrasts it with the predicate-AND product $\fprate_1 \cdot \fprate_2$ of \S6.2 (the "bit-AND is not generally a Bloom filter for $A \cap B$" caveat).
6. Closes with the open question that motivates chapters 14 and 15: what if the two filters do not share a hash family?

**Conventions (matching book notation):**
- `epsilon` ($\fprate$, FPR) = false-positive rate; `omega` ($\fnrate$, FNR) = false-negative rate.
- For each Bloom filter $\omega = 0$ exactly; only the FPR composes non-trivially.
- $m$ = bit-array length; $n$ = inserted items; $k$ = hash functions.
- A single seeded `rng` drives every random draw.

In [1]:
import hashlib
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

print(f"NumPy {np.__version__}")
print(f"Matplotlib {plt.matplotlib.__version__}")

NumPy 2.4.4
Matplotlib 3.10.9


## 1. Bloom Filter Class (Reprise from Chapter 5)

We restate the Bloom filter class from `ch05-approximate-sets.ipynb` so this notebook runs standalone. The hash family is the per-index SHA-256 construction $h_i(x) = \mathrm{SHA256}(i \,\Vert\, x) \bmod m$, taking the first 8 bytes of each digest as a 64-bit integer before reduction. The per-index salt scales to any $k$ without exhausting a fixed digest budget, and the chapter 5 notebook already demonstrated that the practical bias against uniformity sits well below the sampling noise of the experiments we run here.

Crucially for chapter 6: when two Bloom filters use the *same* parameters $(m, k)$ and the *same* hash family $\{h_1, \ldots, h_k\}$, the bit positions a query reads are identical across the two filters. The bit-OR and bit-AND operations of \S6.1 and \S6.2 only make sense under that hash-alignment condition. The open-question cell at the end returns to what happens when alignment fails.

In [2]:
class BloomFilter:
    """Textbook Bloom filter: m bits, k SHA-256-derived hash functions.

    Hash family: h_i(x) = SHA256(str(i) + ':' + str(x)) mod m, for i in 0..k-1.
    Two BloomFilter instances built with the same (m, k) share the same hash
    family by construction, which is the alignment condition the bit-OR and
    bit-AND constructions of chapter 6 require.
    """

    def __init__(self, m: int, k: int) -> None:
        if m <= 0 or k <= 0:
            raise ValueError(f"need m > 0 and k > 0; got m={m}, k={k}")
        self.m = m
        self.k = k
        self.bits = np.zeros(m, dtype=bool)

    def _positions(self, x: object) -> list[int]:
        """Return the k bit positions that x hashes to."""
        key = str(x).encode("utf-8")
        positions = []
        for i in range(self.k):
            digest = hashlib.sha256(str(i).encode("utf-8") + b":" + key).digest()
            positions.append(int.from_bytes(digest[:8], "big") % self.m)
        return positions

    def add(self, x: object) -> None:
        for pos in self._positions(x):
            self.bits[pos] = True

    def __contains__(self, x: object) -> bool:
        return all(self.bits[pos] for pos in self._positions(x))

    def load(self) -> float:
        return float(self.bits.mean())


# Smoke test: empty filter rejects, inserted item is present.
bf_smoke = BloomFilter(m=128, k=4)
assert ("anything" in bf_smoke) is False
bf_smoke.add("hello")
assert ("hello" in bf_smoke)
print("BloomFilter restated; insert and query work as expected.")

BloomFilter restated; insert and query work as expected.


## 2. Two Overlapping Filters

We build $\ASet{B}_1$ from item set $S_1$ and $\ASet{B}_2$ from item set $S_2$, both at $m = 5000$, $k = 7$, $n_1 = n_2 = 500$. The two underlying item sets are drawn from $[0, 50000)$ with overlap, so $A = S_1$ and $B = S_2$ exist as ordinary Python sets and we can ask predicate questions about $A \cup B$ and $A \cap B$ directly without the filter approximation.

At these parameters the per-filter FPR predicted by $(1 - e^{-kn/m})^k$ is $(1 - e^{-0.7})^7 \approx 0.0082$, identical to the standard operating point used in the chapter 5 notebook for cross-comparison.

In [3]:
M = 5000
K = 7
N1 = N2 = 500
UNIVERSE_MAX = 50_000

# Draw S1 and S2 from the same universe so they overlap by chance.
S1 = set(int(x) for x in rng.choice(UNIVERSE_MAX, size=N1, replace=False))
S2 = set(int(x) for x in rng.choice(UNIVERSE_MAX, size=N2, replace=False))

A_union_B = S1 | S2
A_inter_B = S1 & S2

# Build the two Bloom filters with shared (m, k) and therefore the same hash family.
B1 = BloomFilter(m=M, k=K)
for x in S1:
    B1.add(x)
B2 = BloomFilter(m=M, k=K)
for x in S2:
    B2.add(x)

# Per-filter theoretical FPR from the chapter 5 formula.
fpr_1_theory = (1.0 - np.exp(-K * N1 / M)) ** K
fpr_2_theory = (1.0 - np.exp(-K * N2 / M)) ** K

print(f"Two filters at (m, k) = ({M}, {K}), n1 = n2 = {N1}")
print(f"  |S1|            = {len(S1)}")
print(f"  |S2|            = {len(S2)}")
print(f"  |S1 union S2|   = {len(A_union_B)}")
print(f"  |S1 inter S2|   = {len(A_inter_B)}")
print(f"  B1.load()       = {B1.load():.4f}   (theory: 1 - e^(-0.7) = {1 - np.exp(-0.7):.4f})")
print(f"  B2.load()       = {B2.load():.4f}")
print(f"  per-filter theoretical FPR  = (1 - e^(-kn/m))^k = {fpr_1_theory:.5f}")

Two filters at (m, k) = (5000, 7), n1 = n2 = 500
  |S1|            = 500
  |S2|            = 500
  |S1 union S2|   = 993
  |S1 inter S2|   = 7
  B1.load()       = 0.4998   (theory: 1 - e^(-0.7) = 0.5034)
  B2.load()       = 0.5050
  per-filter theoretical FPR  = (1 - e^(-kn/m))^k = 0.00819


## 3. Predicate-OR Union FPR

Proposition 6.1 in the chapter derives the union FPR as a Boolean OR composition of the two operand predicates, exactly the chapter 3 OR-gate composition lifted to the set level:

$$\fprate_\cup \;=\; 1 - (1 - \fprate_1)(1 - \fprate_2).$$

The proof assumes the cross-set error events at every query point are mutually independent. We test the predicate-OR construction directly: for each out-of-set query $x$ we evaluate `(x in B1) or (x in B2)` and count the fraction of $\top$ outputs.

Out-of-set queries are integers drawn from $[0, 100000)$ that lie outside $S_1 \cup S_2$, so any reported $\top$ is a genuine false positive of the union predicate.

In [4]:
N_QUERIES = 10_000
QUERY_MAX = 100_000

# Draw 10000 out-of-set queries: integers in [0, QUERY_MAX) that are NOT in S1 \cup S2.
queries_union: list[int] = []
while len(queries_union) < N_QUERIES:
    candidates = rng.integers(0, QUERY_MAX, size=N_QUERIES)
    for c in candidates:
        ci = int(c)
        if ci not in A_union_B:
            queries_union.append(ci)
            if len(queries_union) == N_QUERIES:
                break

# Predicate-OR: query each filter, then OR the two booleans.
fp_predor = sum(1 for q in queries_union if (q in B1) or (q in B2))
fpr_predor_emp = fp_predor / N_QUERIES

fpr_union_theory = 1.0 - (1.0 - fpr_1_theory) * (1.0 - fpr_2_theory)

print(f"Predicate-OR union: empirical vs closed-form FPR")
print(f"  per-filter theoretical FPR                   = {fpr_1_theory:.5f}")
print(f"  composition formula 1 - (1-eps_1)(1-eps_2)   = {fpr_union_theory:.5f}")
print(f"  empirical false positives                    = {fp_predor} / {N_QUERIES}")
print(f"  empirical predicate-OR FPR                   = {fpr_predor_emp:.5f}")
print(f"  absolute error                               = {abs(fpr_predor_emp - fpr_union_theory):.5f}")
print()
if abs(fpr_predor_emp - fpr_union_theory) < 0.005:
    print("Within +/- 0.005 of the chapter 6 union-FPR formula.")
    print("Predicate-OR composes by the chapter 3 OR-gate rule, lifted to the set level.")
else:
    print("Outside the +/- 0.005 band; the cross-set independence assumption may need a closer look.")

Predicate-OR union: empirical vs closed-form FPR
  per-filter theoretical FPR                   = 0.00819
  composition formula 1 - (1-eps_1)(1-eps_2)   = 0.01632
  empirical false positives                    = 148 / 10000
  empirical predicate-OR FPR                   = 0.01480
  absolute error                               = 0.00152

Within +/- 0.005 of the chapter 6 union-FPR formula.
Predicate-OR composes by the chapter 3 OR-gate rule, lifted to the set level.


## 4. Bit-OR Union FPR and How It Differs

\S6.1 also describes the bit-wise OR realization: take the two bit arrays $b_1$ and $b_2$ and form $b_1 \lor b_2$. The chapter notes that this bit array represents a *superset* of $A \cup B$; every member of $A \cup B$ has all $k$ bits set in $b_1 \lor b_2$, so no false negatives, but the resulting object may report $\top$ on out-of-set queries at a rate higher than the predicate-OR composition suggests.

The two queries are structurally different:

- **Predicate-OR:** $(\,\forall i: b_1[h_i(x)] = 1\,) \;\;\text{OR}\;\; (\,\forall i: b_2[h_i(x)] = 1\,)$. One of the two filters has *all* its bits set; the other one's bits do not matter.
- **Bit-OR:** $\,\forall i: \big(b_1[h_i(x)] = 1 \;\text{OR}\; b_2[h_i(x)] = 1\big)$. At every queried position, *at least one* of the two arrays has the bit set; the responsible array may differ across positions.

Bit-OR is a structurally weaker condition (it allows each queried position to be satisfied by a different array), so it accepts strictly more queries and gives a higher FPR. At $\fprate \approx 0.5$ per-bit load, the OR'd array has load $\approx 1 - 0.5 \cdot 0.5 = 0.75$, and a $k$-position query on a $0.75$-load Bloom filter has FPR $\approx 0.75^7 \approx 0.13$, an order of magnitude above the predicate-OR formula.

In [5]:
# Build the bit-OR filter and query it on the same out-of-set stream.
B_bitor = BloomFilter(m=M, k=K)
B_bitor.bits = B1.bits | B2.bits

fp_bitor = sum(1 for q in queries_union if q in B_bitor)
fpr_bitor_emp = fp_bitor / N_QUERIES

# Theoretical bit-OR FPR from the merged-array load: p_or = 1 - (1-p_1)(1-p_2), then p_or^k.
p_or = 1.0 - (1.0 - B1.load()) * (1.0 - B2.load())
fpr_bitor_theory = p_or ** K

print(f"Bit-OR union: empirical vs predicate-OR formula")
print(f"  B_bitor load                                 = {B_bitor.load():.4f}")
print(f"  bit-OR FPR via merged-array load (p_or^k)    = {fpr_bitor_theory:.5f}")
print(f"  empirical bit-OR FPR                         = {fpr_bitor_emp:.5f}  ({fp_bitor} / {N_QUERIES})")
print(f"  predicate-OR formula 1 - (1-eps_1)(1-eps_2)  = {fpr_union_theory:.5f}")
print()
print(f"  gap (bit-OR empirical vs predicate-OR formula)   = {fpr_bitor_emp - fpr_union_theory:+.5f}")
print(f"  gap (bit-OR empirical vs predicate-OR empirical) = {fpr_bitor_emp - fpr_predor_emp:+.5f}")
print()
print("Bit-OR sits well above predicate-OR. The two queries are different events")
print("on the same array even when the two filters share a hash family; the bit-OR")
print("realization implements a Bloom filter for a superset of A union B, not for")
print("A union B itself. The chapter 6 union FPR formula is the predicate composition.")

Bit-OR union: empirical vs predicate-OR formula
  B_bitor load                                 = 0.7482
  bit-OR FPR via merged-array load (p_or^k)    = 0.13650
  empirical bit-OR FPR                         = 0.13530  (1353 / 10000)
  predicate-OR formula 1 - (1-eps_1)(1-eps_2)  = 0.01632

  gap (bit-OR empirical vs predicate-OR formula)   = +0.11898
  gap (bit-OR empirical vs predicate-OR empirical) = +0.12050

Bit-OR sits well above predicate-OR. The two queries are different events
on the same array even when the two filters share a hash family; the bit-OR
realization implements a Bloom filter for a superset of A union B, not for
A union B itself. The chapter 6 union FPR formula is the predicate composition.


## 5. Bit-AND Intersection-Shaped Object

The bit-wise AND of the two bit arrays produces an object that looks like a Bloom filter, but \S6.2's caveat is explicit: it is *not* in general a Bloom filter for the intersection of the underlying sets. The bit-AND coincides with the predicate intersection on items inserted into *both* sets, but on items inserted into only one set it can still report $\top$ from coincidental hash collisions, and on out-of-set items it inherits hash-pattern artefacts from both arrays.

The predicate-AND rate from \S6.2 is the product of the operand FPRs:

$$\fprate_\cap^{\text{pred}} \;=\; \fprate_1 \cdot \fprate_2.$$

We measure both the predicate-AND FPR (`(q in B1) and (q in B2)`) and the bit-AND FPR (a single query on $b_1 \land b_2$). Out-of-set queries are integers in $[0, 100000)$ not in $S_1 \cap S_2$.

In [6]:
# Construct the bit-AND object via bit-wise AND.
B_bitand = BloomFilter(m=M, k=K)
B_bitand.bits = B1.bits & B2.bits

# Draw 10000 items NOT in S1 \cap S2.
queries_inter: list[int] = []
while len(queries_inter) < N_QUERIES:
    candidates = rng.integers(0, QUERY_MAX, size=N_QUERIES)
    for c in candidates:
        ci = int(c)
        if ci not in A_inter_B:
            queries_inter.append(ci)
            if len(queries_inter) == N_QUERIES:
                break

fp_predand = sum(1 for q in queries_inter if (q in B1) and (q in B2))
fpr_predand_emp = fp_predand / N_QUERIES

fp_bitand = sum(1 for q in queries_inter if q in B_bitand)
fpr_bitand_emp = fp_bitand / N_QUERIES

fpr_predand_theory = fpr_1_theory * fpr_2_theory

# Theoretical bit-AND FPR from the AND'd-array load: p_and = p_1 * p_2 (independence approx).
p_and = B1.load() * B2.load()
fpr_bitand_theory = p_and ** K

print(f"Intersection-shaped queries: bit-AND vs predicate-AND")
print(f"  per-filter theoretical FPR                    = {fpr_1_theory:.5f}")
print(f"  predicate-AND closed form (eps_1 * eps_2)     = {fpr_predand_theory:.6f}")
print(f"  bit-AND via merged-array load (p_1 p_2)^k     = {fpr_bitand_theory:.6f}")
print()
print(f"  empirical predicate-AND FPR                   = {fpr_predand_emp:.5f}  ({fp_predand} / {N_QUERIES})")
print(f"  empirical bit-AND FPR                         = {fpr_bitand_emp:.5f}  ({fp_bitand} / {N_QUERIES})")
print()
print(f"  gap (bit-AND empirical vs eps_1 * eps_2)          = {fpr_bitand_emp - fpr_predand_theory:+.6f}")
print(f"  gap (bit-AND empirical vs predicate-AND empirical) = {fpr_bitand_emp - fpr_predand_emp:+.6f}")
print()
print("At eps_1 * eps_2 ~ 6.7e-5 the expected predicate-AND count over 10k queries is")
print("under one, so the empirical estimate sits in Poisson-noise territory rather")
print("than mean-error territory. The bit-AND number is the one with structural content:")
print("it can deviate from the predicate-AND formula even at small expected counts,")
print("and the discrepancy is the section 6.2 caveat made empirical.")

Intersection-shaped queries: bit-AND vs predicate-AND
  per-filter theoretical FPR                    = 0.00819
  predicate-AND closed form (eps_1 * eps_2)     = 0.000067
  bit-AND via merged-array load (p_1 p_2)^k     = 0.000065

  empirical predicate-AND FPR                   = 0.00020  (2 / 10000)
  empirical bit-AND FPR                         = 0.00020  (2 / 10000)

  gap (bit-AND empirical vs eps_1 * eps_2)          = +0.000133
  gap (bit-AND empirical vs predicate-AND empirical) = +0.000000

At eps_1 * eps_2 ~ 6.7e-5 the expected predicate-AND count over 10k queries is
under one, so the empirical estimate sits in Poisson-noise territory rather
than mean-error territory. The bit-AND number is the one with structural content:
it can deviate from the predicate-AND formula even at small expected counts,
and the discrepancy is the section 6.2 caveat made empirical.


## 6. The Bit-AND vs Predicate-AND Distinction

The chapter 6 \S6.2 closing caveat is the rule: bit-AND of two Bloom filters is *not* in general a Bloom filter for $A \cap B$. The two queries are structurally different:

- **Predicate-AND** queries each filter separately and ANDs the two Boolean answers. By cross-set independence on out-of-set queries, the joint false-positive rate is $\fprate_1 \cdot \fprate_2$, the chapter 3 AND-gate composition lifted to the set level. At the operating point $\fprate \approx 0.008$ the product is $\sim 6.7 \times 10^{-5}$, an extremely tight intersection rate that pays off the chapter 6 promise: composition shrinks error.
- **Bit-AND** queries a single filter whose bit array is $b_1 \land b_2$. A query for $x \notin A \cap B$ reports $\top$ when every one of $x$'s $k$ hash positions is set in *both* arrays simultaneously. The chapter notes two failure modes: bit-AND can be more permissive than the predicate intersection (a position set in both arrays for unrelated reasons still satisfies the AND), and on items inserted into only one set it can still erroneously report $\top$.

**Direction of the deviation.** Bit-AND queries a Bloom filter with load $\approx p_1 p_2$, giving FPR $\approx (p_1 p_2)^k$. At $p_1 = p_2 \approx 0.5$ this is $0.25^7 \approx 6.1 \times 10^{-5}$, which happens to be close to $(p_1^k)(p_2^k) = (\fprate_1)(\fprate_2)$ for the merged-array model. The deviation from the predicate formula in this regime is small. At other regimes (small $m$, large $k$, or asymmetric loads) the gap can be either positive or negative depending on which hash-pattern artefact dominates.

Both predicate-AND and bit-AND are honest about the same underlying noise. They differ because they ask different questions. The chapter's stance, repeated here: composition of approximate sets is a predicate-level operation. The bit-array shortcut is genuinely useful for the union (where bit-OR is at least an upper bound on the predicate-OR rate) but does not enjoy a clean rate formula for the intersection.

## 7. Open Question: What If the Two Filters Use Different Hash Families?

Everything in \S6.1 and \S6.2 assumed that $\ASet{B}_1$ and $\ASet{B}_2$ share their hash family $\{h_1, \ldots, h_k\}$. That alignment is what makes bit-OR and bit-AND meaningful: a queried bit position is the *same* position in both arrays, so a bit set in $b_1$ at position $h_i(x)$ is comparable with a bit set in $b_2$ at the same position.

Drop that assumption and the operations stop being interpretable. If $\ASet{B}_2$ uses a different hash family, then $h_i^{(2)}(x)$ lands at a different array position than $h_i^{(1)}(x)$, and the bit at position $j$ in $b_2$ says nothing about whether $x$ would be reported present by $\ASet{B}_2$. Bit-OR and bit-AND become arithmetic on uncorrelated bit patterns, with no probabilistic relationship to $A \cup B$ or $A \cap B$.

The predicate-level union and intersection of \S6.1 and \S6.2 still apply: query each filter separately, then OR or AND the Boolean answers. That construction is hash-family-independent, and the rate formulas $1 - (1 - \fprate_1)(1 - \fprate_2)$ and $\fprate_1 \cdot \fprate_2$ stay correct under the cross-set independence assumption (which is in fact *cleaner* with independent hash families, since the events have no shared randomness through the hash). The operational shortcut is what disappears.

Two later chapters return to this:

- **Chapter 14** (Perfect-Hash Filters as Bernoulli Hash Functions) introduces a construction where the hash structure is itself parameterized by the inserted set $A$. Two BHF instances for two different sets do not share hash structure by construction, so bit-array operations between them are not defined; composition is exclusively at the predicate level.
- **Chapter 15** (Bernoulli Perfect Hash Filters) gives an alignment-aware construction that supports compositional operations between filters built over related sets, while still attaining the entropy lower bound that the Bloom filter misses by the $1/\ln 2 \approx 1.44$ factor.

What this notebook has shown: predicate-OR composes by the chapter 3 OR-gate formula and the empirical FPR sits within sampling noise of $1 - (1 - \fprate_1)(1 - \fprate_2)$; bit-OR sits well above that rate because the bit-array realization represents a superset of $A \cup B$, not $A \cup B$ itself; bit-AND is *not* the same object as the predicate intersection, and the chapter 6 \S6.2 caveat is the rule. The clean composition rules of chapter 6 are predicate-level rules. Bit-array shortcuts work asymmetrically: bit-OR upper-bounds the predicate union and is still useful as a streaming construction; bit-AND has no comparable guarantee, and the failure mode is exactly the one chapters 14 and 15 are designed to address.

---

## Closing: Cross-References and Next Steps

**Chapter 6** lifts the per-bit Boolean composition of chapter 3 to the per-set level: union, intersection, complement, and difference each inherit their rate formulas from the corresponding Boolean gate. \S6.1 is the union and \S6.2 is intersection, complement, and difference, both stated at the predicate level. \S6.3 walks the cancellation phenomena that take a one-sided filter off the perfect-filter line, and \S6.4 closes the chapter with the closure-of-the-axis question that motivates chapters 14 and 15.

**Chapter 7** (Classification Measures) attaches PPV, NPV, and Youden's $J$ to the composed rates of chapter 6, so a downstream user can read off the practical accuracy of a composed filter, not just its raw FPR.

**Chapter 14** (Optimal Bernoulli Sets) and **Chapter 15** (Perfect-Hash Filters) close the $1/\ln 2$ inefficiency gap of the Bloom filter, and in doing so replace bit-array operations with composition rules that work on the predicate level by construction.

**Summary of what this notebook verified:**

| Section | Claim | Verified |
|---------|-------|----------|
| \S6.1 predicate-OR | Empirical predicate-OR FPR matches $1 - (1 - \fprate_1)(1 - \fprate_2)$ within $\pm 0.005$ | Yes |
| \S6.1 bit-OR | Bit-OR FPR sits well above the predicate formula; bit-OR is a Bloom filter for a *superset* of $A \cup B$ | Yes (gap is visible) |
| \S6.2 predicate-AND | Predicate-AND empirical FPR consistent with $\fprate_1 \cdot \fprate_2$ within Poisson noise | Yes (small expected count) |
| \S6.2 bit-AND caveat | Bit-AND is not generally a Bloom filter for $A \cap B$; numerically distinct from the predicate intersection | Yes |
| Hash-family alignment | Open question: bit operations meaningless if hash families differ; forward to ch 14 and 15 | Open |